# Build a CNN for MNIST

**Theme:**

*“Let the network learn features, not you.”*

## Imports

In [1]:
import torch 
import torch.nn as nn


torch.__version__

'2.8.0+cu129'

In [3]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## Define the CNN

In [ ]:
# Data
transform = transforms.ToTensor()

train_data = datasets.MNIST(root='../week2/data', download=True, train=True, transform=transform)
test_data = datasets.MNIST(root='../week2/data', download=True, train=False, transform=transform) 

train_loader = DataLoader(train_data, batch_size=100, shuffle=True) 
test_loader = DataLoader(test_data, batch_size=100, shuffle=False) 

# Network
class MyCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, stride=1),  # 1(black&white)28x28x1 -> 28x28x32 *formula:- (width - filter + 2 x pdding)/stride + 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),                                # 28x28x32 -> 14x14x32

            nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=1), # 14x14x32 => 14x14x64
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)                                   # 14x14 => 7x7
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 512),
            nn.ReLU(),
            nn.Linear(512,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )
    
    def forward(self,x):
        x = self.conv(x)
        output = self.fc(x)
        return output 


In [25]:
# model, loss function, optimizer
model = MyCNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) 

## Train CNN

In [26]:
images, labels = next(iter(train_loader))
images.shape, labels.shape

(torch.Size([100, 1, 28, 28]), torch.Size([100]))

### Training

In [36]:
max_epochs = 10

for epoch in range(max_epochs):
    # Training
    model.train()

    total_loss = 0.0

    for images, labels in train_loader:
        # forward prop
        outputs = model(images)
        # compute loss
        loss = loss_fn(outputs,labels) 

        # backward prop
        loss.backward()
        # weights update
        optimizer.step()
        # zero grad
        optimizer.zero_grad()

        total_loss += loss.item() 

    print(f"Epoch: {epoch+1} | Loss: {total_loss/len(train_loader):.6f}")

Epoch: 1 | Loss: 0.007938289074019545
Epoch: 2 | Loss: 0.006068751000727086
Epoch: 3 | Loss: 0.007024263978830731
Epoch: 4 | Loss: 0.006638557209162931
Epoch: 5 | Loss: 0.004336219195672584
Epoch: 6 | Loss: 0.004456571319271158
Epoch: 7 | Loss: 0.00390804871797144
Epoch: 8 | Loss: 0.005368124078840613
Epoch: 9 | Loss: 0.004134076465094552
Epoch: 10 | Loss: 0.003139916746989873


### Evaluation

In [37]:
# Evaluation
model.eval()

correct_pred = 0
total_pred = 0 
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images) 
        loss = loss_fn(outputs,labels) 

        predictions = torch.argmax(outputs, dim=1)
        correct_pred += (predictions==labels).sum().item()
        total_pred += labels.shape[0] 

acc = correct_pred/total_pred * 100
print("Test Accuracy:",round(acc,2),"%")

Test Accuracy: 99.05 %


## Comparison with MLP

- Using ANN we get test accuracy of **97.62 %**
  
- Using CNN we get test accuracy of **99.01 %** 

📌 Answer:

- What accuracy did CNN get?
- What accuracy did MLP get?
- Which is better?
- Why?

## Inspect Feature Maps (Visual Intuition)

In [35]:
features = model.conv(images) 
print(features.shape)

torch.Size([100, 64, 7, 7])


📌 Answer:

- What does this shape mean?

- What is 100? What is 64?

# Interaction

### big realization moment


MLP:

- Learned from flattened pixels

CNN:

- Learned edges → shapes → digit structure

This is why CNN wins. (Accuracy: **99% vs 97%**)